In [7]:
import os
from dotenv import load_dotenv

load_dotenv()  
api_key = os.getenv("OPENROUTER_API_KEY")

model = os.getenv("MODEL")

In [8]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openrouter import ChatOpenRouter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import InMemoryVectorStore


In [9]:
loader = PyMuPDFLoader(file_path="../data/CorporatePolicyHandbook.pdf")
docs = loader.load()


In [10]:
splitter = RecursiveCharacterTextSplitter(chunk_size=600,chunk_overlap=50)
docs = splitter.split_documents(docs)

In [11]:
embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2",
)

c:\Users\akash\OneDrive\Desktop\Rag\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1692.78it/s]


In [12]:
vector_store = InMemoryVectorStore.from_documents(
    embedding=embeddings,
    documents=docs
)

In [13]:
llm = ChatOpenRouter(
    api_key=api_key,
    model=model
)

In [14]:
query = "Is there any crietria for creating User credentials"
chunks = vector_store.similarity_search(query)
# print(chunks[0].page_content)

context = ""

for chunk in chunks:
    context += chunk.page_content + "\n\n"

print(context)

User credentials represent the primary defensive line of enterprise infrastructure. Passwords must be
at least 14 characters long and combine uppercase characters, lowercase characters, numerical
digits, and approved special symbols. Passwords must not be reused across three consecutive
cycles or shared across external consumer accounts. Multi-Factor Authentication (MFA) is strictly
mandatory across all enterprise access gateways, with physical FIDO2 security keys required for
high-privilege system administrators.
Corporate Governance, Risk & Compliance Manual — Document Ref: POL-GRP-2026-V4

Global Ethics Hotline: 1-800-555-ETHIC (Toll-free, 24/7/365, operated by an independent third-
party monitoring agency).
Secure Compliance Portal: Web-based encrypted submission portal with anonymous bidirectional
messaging capabilities.
Direct Written Notification: Confidential correspondence directed to the Chair of the Audit and
Governance Committee, General Counsel Office.
5.3 Formal Acknowled

In [15]:
prompt = f""" 

 You are Smart Rag system. You always provide response for user question based on the provide context.

 Question : {query}


 Context : {context}

"""

res = llm.invoke(prompt)

In [16]:
print(res.content)

Based on the provided context (Section 2.3 Identity Governance, Credentialing & Password Requirements), yes, there are specific criteria for creating user credentials:

*   **Length:** Passwords must be at least **14 characters** long.
*   **Complexity:** They must combine **uppercase characters, lowercase characters, numerical digits, and approved special symbols**.
*   **Reuse:** Passwords must not be reused across **three consecutive cycles**.
*   **Sharing:** Passwords must not be shared across **external consumer accounts**.

Additionally, the manual states that **Multi-Factor Authentication (MFA) is strictly mandatory** across all enterprise access gateways.


## Multi Document RAG

In [17]:
from langchain_community.document_loaders import WebBaseLoader

In [18]:
web_loader = WebBaseLoader(web_path="https://qdrant.tech/documentation/overview/")

In [19]:
web_data = web_loader.load()

In [20]:
web_chunks = splitter.split_documents(web_data)

In [ ]:
vector_store.add_documents(web_chunks)

In [27]:
# query = "How Qdrant Cloud filtering works"
query = "Is there any crietria for creating User credentials"
chunks = vector_store.similarity_search(query)
# print(chunks[0].page_content)

context = ""

for chunk in chunks:
    context += chunk.page_content + "\n\n"


In [ ]:
print(context)

In [29]:
prompt = f""" 

 You are Smart Rag system. You always provide response for user question based on the provide context.

 Question : {query}


 Context : {context}

"""

res = llm.invoke(prompt)

In [ ]:
print(res.content)